# Liang et al. MAD — Prover → Verifier → Judge

A sequential three-step debate system based on Liang et al. (2023) *Encouraging Divergent Thinking in LLMs through Multi-Agent Debate*.

**Three agents:**
- **Prover** (Optimistic Advocate): Makes the strongest case for hypothesis novelty, searches for supporting literature
- **Verifier** (Conservative Skeptic): Challenges the Prover's arguments, searches for counter-evidence or prior work
- **Judge** (Neutral Arbiter): Synthesizes arguments from both sides, outputs a structured verdict; exits early if a verdict is reached

**Execution order (per round):**
```
Prover → Verifier → Judge
                     └── final_verdict non-empty → exit
                     └── final_verdict empty → next round
```
Runs for at most `max_rounds` rounds; if still not converged, summons an independent Judge for a final verdict (Liang et al.'s *ultimate technique*).

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().parent.absolute()))

import os
import json
import re
import asyncio
from typing import Dict, List, Optional

os.environ["SEMANTIC_SCHOLAR_API_KEY"] = ""
os.environ["OPENAI_API_KEY"] = ""
os.environ["NCBI_API_KEY"] = ""

## Agent System Prompts

In [2]:
PROVER_SYSTEM = """\
You are the PROVER in a sequential scientific debate about research hypothesis novelty.

YOUR ROLE: Make the strongest possible case for why this hypothesis is BOTH novel and biologically/scientifically plausible.

YOUR STRATEGY:
1. Use your literature search tool (1-3 targeted queries) to find SUPPORTING evidence.
2. THE "NOVELTY GAP" RULE (CRITICAL): If you cannot find direct literature connecting the components of your hypothesis, you are STRICTLY FORBIDDEN from simply arguing "this represents a research gap therefore it is novel."
3. PROVE PLAUSIBILITY: When direct evidence is absent, you MUST defend the biological/mechanistic plausibility of your idea. Use indirect evidence, parallel pathways, homologous structures, or established principles from related fields to logically deduce why your proposed mechanism would work.
4. Distinguish this hypothesis from the closest prior work.
5. Counter the Verifier's challenges with specific, deduced evidence, not just rhetoric.

Output: Plain argumentative text (no JSON). Be specific, cite indirect/direct search results, and clearly map out your logical deductions.
"""

VERIFIER_SYSTEM = """\
You are the VERIFIER in a sequential scientific debate about research hypothesis novelty.

YOUR ROLE: Rigorously challenge both the novelty and the PLAUSIBILITY of the Prover's claims.

YOUR STRATEGY:
1. DECOMPOSE & SEARCH: Do not just search for the exact, full hypothesis. Use your literature search tool (1-3 targeted queries) to decompose it into pairs (e.g., Molecule X + Mechanism Y, or Mechanism Y + Phenotype Z).
2. ATTACK THE MISSING LINK (CRITICAL): If there is no direct prior work, DO NOT just say "I found no studies." Instead, evaluate the biological/physical plausibility of the Prover's deduction. Are there known behaviors of these individual components that logically CONTRADICT the proposed connection?
3. Scenarios:
   A) SPECIOUS/IMPLAUSIBLE: Literature on isolated components strongly suggests they would NOT interact as proposed → challenge the biological viability strongly.
   B) NOVEL BUT PLAUSIBLE: No direct evidence, but the underlying physics/biology of the separate parts could theoretically support the link → acknowledge plausibility but point out the lack of empirical proof.
4. Identify the most similar existing work and explain precisely how the new hypothesis overlaps or diverges.

Output: Plain argumentative text (no JSON). Be specific, cite your decomposed search results, and attack weak logical leaps.
"""

JUDGE_SYSTEM = """\
You are the JUDGE in a sequential scientific debate about research hypothesis novelty.

YOUR ROLE: Deliver an impartial, rigorously comprehensive verdict based on the quality of arguments from both sides.

YOUR STANDARDS & RULES:
1. WEIGH EVIDENCE: Judge based on evidence quality and biological plausibility, not rhetorical force.
2. NOVELTY PARADOX:
   - If Verifier found a DIRECT CONTRADICTION with citation → lower score (≤4).
   - If Verifier found only "absence of evidence" (no papers) AND Prover successfully defended its biological plausibility → protect novelty (7-9).
   - If Prover failed to provide a plausible logical deduction for the missing link → lower score (≤5).
3. COMPREHENSIVENESS CHECK (CRITICAL):
   - You MUST extract all distinct components of the original hypothesis (e.g., specific molecules, multiple mechanisms like migration/immunity, spatial phenotypes, or physics concepts).
   - The Prover is NOT allowed to cherry-pick. They must defend the novelty and plausibility of the ENTIRE hypothesis, not just a fragment of it.
4. PENALTY RULE:
   - MISSING COMPONENTS: If the Prover dodges or fails to address core components of the hypothesis, you MUST cap the final score at 6 (REVISE).
   - ANTI-EVASION (SEVERE): If the Prover silently drops, ignores, or alters the most controversial/implausible components of the original hypothesis (e.g., ignoring "quantum entanglement" to focus only on standard neuroscience) to make it easier to defend, this is considered scientific misconduct. You MUST immediately issue a REJECT verdict with a score of ≤3.
5. SCORE RANGE: 1-10 (≥7.5: APPROVE, 5-7.4: REVISE, <5: REJECT).

OUTPUT FORMAT — always output exactly one JSON block:

If debate should CONTINUE (arguments still evolving, or Prover missed components):
```json
{"final_verdict": "", "continue_reason": "<Why more debate is needed. Explicitly list any hypothesis components the Prover ignored and command them to address them>"}
If you can make a FINAL DECISION (max rounds reached or consensus achieved):

JSON
{
  "final_verdict": "APPROVE" | "REVISE" | "REJECT",
  "score": <int 1-10>,
  "confidence": <float 0-1>,
  "rationale": "<Your reasoning, specifically noting if the Prover attempted to evade absurd claims>",
  "unaddressed_components": ["<List any original hypothesis claims the Prover completely ignored or dropped>"],
  "prover_wins": ["<claims that survived>"],
  "verifier_wins": ["<claims that were refuted>"],
  "verifier_had_real_contradiction": <true|false>
}
```
"""

## Create Agents

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from HypothesisGeneration.agents.multi_critic_coordinator import (
    MultiCriticCoordinator,
    _create_openai_client,
    _extract_usage,
)

# Directly instantiate the inner Coordinator to access _create_search_tool
_coordinator = MultiCriticCoordinator(base_model="gpt-4o")

def _make_agent(name: str, system: str, sources: List[str], temp: float) -> AssistantAgent:
    client = _create_openai_client("gpt-4o", temperature=temp)
    tool   = _coordinator._create_search_tool(sources)
    return AssistantAgent(
        name=name,
        model_client=client,
        system_message=system,
        max_tool_iterations=10,
        reflect_on_tool_use=True,
        tools=[tool],
    )

prover   = _make_agent("Prover",   PROVER_SYSTEM,   ["semantic_scholar", "biorxiv", "arxiv"],   temp=0.9)
verifier = _make_agent("Verifier", VERIFIER_SYSTEM, ["pubmed", "europepmc"],                    temp=0.3)
judge    = _make_agent("Judge",    JUDGE_SYSTEM,    ["pubmed", "semantic_scholar", "crossref"],  temp=0.5)

print("Agents created: Prover / Verifier / Judge")

## Debate Engine

In [4]:
def _parse_judge(text: str) -> Dict:
    """Extract the JSON block from Judge's response."""
    m = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    # Fallback: bare JSON
    try:
        s = text.find("{")
        e = text.rfind("}") + 1
        if s != -1 and e > s:
            return json.loads(text[s:e])
    except json.JSONDecodeError:
        pass
    return {"final_verdict": "", "continue_reason": "parse_failed"}


def _build_prover_task(hypothesis: str, round_idx: int,
                       prover_history: List[str], verifier_history: List[str]) -> str:
    if round_idx == 0:
        return (
            f"## Hypothesis:\n{hypothesis}\n\n"
            "## Your Task:\n"
            "Search the literature and make the strongest possible case for why this hypothesis "
            "is novel and scientifically significant. Be specific about mechanisms and cite evidence."
        )
    verifier_latest = verifier_history[-1][:1500] if verifier_history else "(none yet)"
    own_history = "\n\n".join(
        f"[Round {i}]:\n{t[:600]}" for i, t in enumerate(prover_history)
    )
    return (
        f"## Hypothesis:\n{hypothesis}\n\n"
        f"## Your Previous Arguments:\n{own_history}\n\n"
        f"## Verifier's Latest Challenge (Round {round_idx - 1}):\n{verifier_latest}\n\n"
        "## Your Task:\n"
        "Rebut the Verifier's challenge with specific evidence. "
        "Search for new supporting literature if needed."
    )


def _build_verifier_task(hypothesis: str, round_idx: int,
                         prover_history: List[str], verifier_history: List[str]) -> str:
    prover_latest = prover_history[-1][:1500] if prover_history else "(none yet)"
    own_history = "\n\n".join(
        f"[Round {i}]:\n{t[:600]}" for i, t in enumerate(verifier_history)
    )
    history_block = f"## Your Previous Challenges:\n{own_history}\n\n" if own_history else ""
    return (
        f"## Hypothesis:\n{hypothesis}\n\n"
        f"{history_block}"
        f"## Prover's Latest Argument (Round {round_idx}):\n{prover_latest}\n\n"
        "## Your Task:\n"
        "Challenge the Prover's claims. Search for prior work or direct contradictions. "
        "Be rigorous but fair — distinguish real contradictions from mere absence of evidence."
    )


def _build_judge_task(hypothesis: str, round_idx: int,
                      prover_history: List[str], verifier_history: List[str],
                      force_continue: bool = False) -> str:
    prover_block   = "\n\n".join(f"[Round {i}]:\n{t[:800]}" for i, t in enumerate(prover_history))
    verifier_block = "\n\n".join(f"[Round {i}]:\n{t[:800]}" for i, t in enumerate(verifier_history))

    if force_continue:
        verdict_instruction = (
            "⚠️ MANDATORY: This is Round 0. The Prover has NOT yet had a chance to rebut the "
            "Verifier's challenges. You MUST output the 'continue' format — do NOT issue a final verdict yet:\n"
            "```json\n{\"final_verdict\": \"\", \"continue_reason\": \"<what unresolved points need a rebuttal round>\"}\n```"
        )
    else:
        verdict_instruction = (
            "Deliver your verdict. If the debate is clearly resolved, output a final decision. "
            "If key points remain unresolved and another round would help, continue. "
            "Always output exactly one JSON block as specified in your system prompt."
        )

    return (
        f"## Hypothesis:\n{hypothesis}\n\n"
        f"## Prover's Arguments (all rounds):\n{prover_block}\n\n"
        f"## Verifier's Arguments (all rounds):\n{verifier_block}\n\n"
        f"## Round {round_idx} Assessment:\n"
        f"{verdict_instruction}"
    )


async def run_liang_debate(
    hypothesis: str,
    max_rounds: int = 3,
    min_rounds: int = 2,      # Prover must rebut at least once before Judge can decide
    verbose: bool = True,
) -> Dict:
    """
    Sequential Prover → Verifier → Judge debate (Liang et al. 2023 style).

    min_rounds: minimum number of full Prover→Verifier→Judge cycles before
                the Judge is allowed to issue a final verdict (default=2,
                so Prover always gets at least one rebuttal round).
    """
    prover_history:   List[str] = []
    verifier_history: List[str] = []
    debate_log:       List[Dict] = []
    usage = {"prompt_tokens": 0, "completion_tokens": 0}
    verdict = {}

    def _log(msg):
        if verbose: print(msg)

    _log("")
    _log("=" * 80)
    _log("LIANG et al. MAD — Prover → Verifier → Judge")
    _log("=" * 80)
    _log(f"Hypothesis: {hypothesis[:120]}..." if len(hypothesis) > 120 else f"Hypothesis: {hypothesis}")
    _log(f"Max rounds: {max_rounds}  |  Min rounds before verdict: {min_rounds}")

    for round_idx in range(max_rounds):
        _log("")
        _log("=" * 80)
        _log(f"ROUND {round_idx}")
        _log("=" * 80)
        round_record = {"round": round_idx, "prover": "", "verifier": "", "judge": {}, "verdict": ""}

        # ── Step 1: Prover ────────────────────────────────────────────────────
        _log("\n[Prover] arguing...")
        prover_task = _build_prover_task(hypothesis, round_idx, prover_history, verifier_history)
        prover_resp = await prover.run(task=prover_task)
        prover_text = prover_resp.messages[-1].content if prover_resp.messages else ""
        prover_history.append(prover_text)
        round_record["prover"] = prover_text
        tok = _extract_usage(prover_resp)
        usage["prompt_tokens"]     += tok.get("prompt_tokens", 0)
        usage["completion_tokens"] += tok.get("completion_tokens", 0)
        _log(f"[Prover] ({len(prover_text)} chars)")
        _log("-" * 40)
        _log(prover_text[:600])

        # ── Step 2: Verifier ──────────────────────────────────────────────────
        _log("\n[Verifier] challenging...")
        verifier_task = _build_verifier_task(hypothesis, round_idx, prover_history, verifier_history)
        verifier_resp = await verifier.run(task=verifier_task)
        verifier_text = verifier_resp.messages[-1].content if verifier_resp.messages else ""
        verifier_history.append(verifier_text)
        round_record["verifier"] = verifier_text
        tok = _extract_usage(verifier_resp)
        usage["prompt_tokens"]     += tok.get("prompt_tokens", 0)
        usage["completion_tokens"] += tok.get("completion_tokens", 0)
        _log(f"[Verifier] ({len(verifier_text)} chars)")
        _log("-" * 40)
        _log(verifier_text[:600])

        # ── Step 3: Judge ─────────────────────────────────────────────────────
        # Before min_rounds, Judge is forced to continue (Prover needs rebuttal chance)
        force_continue = (round_idx < min_rounds - 1)
        _log(f"\n[Judge] deliberating... {'(forced continue — min_rounds not reached)' if force_continue else ''}")
        judge_task = _build_judge_task(hypothesis, round_idx, prover_history, verifier_history,
                                       force_continue=force_continue)
        judge_resp = await judge.run(task=judge_task)
        judge_text = judge_resp.messages[-1].content if judge_resp.messages else ""
        verdict    = _parse_judge(judge_text)

        # Code-level enforcement: override any early verdict before min_rounds
        if force_continue and verdict.get("final_verdict", ""):
            _log(f"   ⚠️  Judge tried to decide early — overriding (min_rounds={min_rounds})")
            verdict = {"final_verdict": "", "continue_reason": f"min_rounds={min_rounds} not reached"}

        round_record["judge"]   = verdict
        round_record["verdict"] = verdict.get("final_verdict", "")
        tok = _extract_usage(judge_resp)
        usage["prompt_tokens"]     += tok.get("prompt_tokens", 0)
        usage["completion_tokens"] += tok.get("completion_tokens", 0)
        debate_log.append(round_record)

        _log(f"[Judge] verdict={verdict.get('final_verdict', '(continue)')}  "
             f"score={verdict.get('score', '?')}  "
             f"confidence={verdict.get('confidence', '?')}")

        if verdict.get("final_verdict", ""):
            _log(f"   ✅ CONVERGENCE at Round {round_idx}")
            break
        else:
            _log(f"   Continue reason: {verdict.get('continue_reason', '?')}")

    # ── Ultimate fallback ─────────────────────────────────────────────────────
    if not verdict.get("final_verdict", ""):
        _log("")
        _log("=" * 80)
        _log("ULTIMATE FALLBACK — Final Judge call")
        _log("=" * 80)
        fallback_task = (
            _build_judge_task(hypothesis, max_rounds, prover_history, verifier_history)
            + "\n\nThis is the FINAL round. You MUST output a definitive verdict (no empty final_verdict)."
        )
        fallback_resp = await judge.run(task=fallback_task)
        fallback_text = fallback_resp.messages[-1].content if fallback_resp.messages else ""
        verdict = _parse_judge(fallback_text)
        tok = _extract_usage(fallback_resp)
        usage["prompt_tokens"]     += tok.get("prompt_tokens", 0)
        usage["completion_tokens"] += tok.get("completion_tokens", 0)
        debate_log.append({"round": "fallback", "judge": verdict, "verdict": verdict.get("final_verdict", "REVISE")})
        _log(f"[Fallback Judge] verdict={verdict.get('final_verdict')}  score={verdict.get('score')}")

    # ── Summary ───────────────────────────────────────────────────────────────
    final_verdict = verdict.get("final_verdict", "REVISE")
    final_score   = verdict.get("score", 5)
    final_conf    = verdict.get("confidence", 0.5)

    _log("")
    _log("=" * 80)
    _log("DEBATE CONCLUDED")
    _log(f"   Verdict:    {final_verdict}")
    _log(f"   Score:      {final_score}/10")
    _log(f"   Confidence: {final_conf}")
    _log(f"   Rounds run: {len(debate_log)}")
    _log(f"   Rationale:  {verdict.get('rationale', '')[:200]}")
    _log("=" * 80)

    return {
        "method": "liang_mad_sequential",
        "final_verdict": final_verdict,
        "score": final_score,
        "confidence": final_conf,
        "rationale": verdict.get("rationale", ""),
        "prover_wins": verdict.get("prover_wins", []),
        "verifier_wins": verdict.get("verifier_wins", []),
        "verifier_had_real_contradiction": verdict.get("verifier_had_real_contradiction", False),
        "rounds": len(debate_log),
        "debate_log": debate_log,
        "usage": usage,
    }

print("Debate engine ready.")

Debate engine ready.


## Run Debate

In [5]:
hypothesis = (
    "Non-invasive scalp Electroencephalography (EEG) theta-wave neurofeedback directly modulates the spatial remodeling of the Extracellular Matrix (ECM) in distal breast tumor microenvironments through macroscopic quantum entanglement of circulating tumor DNA (ctDNA) and vagus nerve-mediated epigenetic resonance."
)

result = await run_liang_debate(hypothesis, max_rounds=3)


LIANG et al. MAD — Prover → Verifier → Judge
Hypothesis: Non-invasive scalp Electroencephalography (EEG) theta-wave neurofeedback directly modulates the spatial remodeling of th...
Max rounds: 3  |  Min rounds before verdict: 2

ROUND 0

[Prover] arguing...
🔍 Searching semantic_scholar...🔍 Searching semantic_scholar...
🔑 Using Semantic Scholar API key (ends with: ...4Av0)

🔑 Using Semantic Scholar API key (ends with: ...4Av0)
🔍 Searching semantic_scholar...
🔑 Using Semantic Scholar API key (ends with: ...4Av0)
🔍 Searching biorxiv...
🔍 Searching biorxiv...
🔍 Searching biorxiv...
🔍 Searching arxiv...🔍 Searching arxiv...

🔍 Searching arxiv...
[Prover] (2991 chars)
----------------------------------------
The hypothesis that EEG theta-wave neurofeedback could modulate ECM remodeling in breast tumor microenvironments via quantum entanglement and vagus nerve-mediated epigenetic resonance carries significant novelty and plausibility, supported by various indirect evidence:

### Novelty of th

## Result Summary

In [6]:
print(f"Verdict:    {result['final_verdict']}")
print(f"Score:      {result['score']}/10")
print(f"Confidence: {result['confidence']}")
print(f"Rounds:     {result['rounds']}")
print(f"\nRationale:\n{result['rationale']}")
print(f"\nProver wins:   {result['prover_wins']}")
print(f"Verifier wins: {result['verifier_wins']}")
print(f"\nTokens used: prompt={result['usage']['prompt_tokens']}  completion={result['usage']['completion_tokens']}")

Verdict:    REJECT
Score:      3/10
Confidence: 0.8
Rounds:     2

Rationale:
The Prover failed to provide a plausible logical deduction or empirical evidence supporting the connection between EEG theta-wave neurofeedback and ECM remodeling in breast tumor environments. The Prover also did not adequately address the highly speculative nature of macroscopic quantum entanglement and vagus nerve-mediated epigenetic resonance in this context. The hypothesis remains largely theoretical, with no direct studies or substantial evidence backing the proposed mechanisms. Furthermore, the Prover did not address the quantum entanglement aspect in Round 1, which is a critical component of the hypothesis.

Prover wins:   []
Verifier wins: ['EEG theta-wave neurofeedback and ECM remodeling', 'Quantum entanglement and vagus nerve influence on tumor microenvironment']

Tokens used: prompt=45148  completion=3005
